In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import xarray as xr
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import cartopy.feature as cfeature
import os
from sklearn.linear_model import LinearRegression
from scipy.stats import gaussian_kde
import xesmf as xe

In [7]:
home_path = os.path.expanduser("~")

path = '/DataFiles'
path2 = '/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - MacBook Air/Python/Ice Cores/data/model/ccsm4_last_millenium'

era5 = xr.open_dataset(home_path + path + "/ae3baa6a74f0aa315dc3de6f83298f0e.nc")
racmo = xr.open_dataset(home_path + path + "/smb_monthlyS_ANT27_ERA5-3H_RACMO2.3p2_197901_202212.nc")
mask = xr.open_dataset(home_path + path + "/TotIS_RACMO_ANT27_IMBIE2.nc")

In [18]:
print(era5["tp"])

<xarray.DataArray 'tp' (valid_time: 560, latitude: 301, longitude: 3600)> Size: 2GB
[606816000 values with dtype=float32]
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 4kB 1979-01-01 ... 2025-08-01
  * latitude    (latitude) float64 2kB -60.0 -60.1 -60.2 ... -89.8 -89.9 -90.0
  * longitude   (longitude) float64 29kB -180.0 -179.9 -179.8 ... 179.8 179.9
    expver      (valid_time) <U4 9kB ...
Attributes: (12/31)
    GRIB_paramId:                             228
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      1083600
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_gridType:                            regular_ll
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                               m
    long_name:                                Total precipitation
    units:          

In [8]:
def compute_2d_bounds(arr):
    """
    Compute approximate 2D cell corner bounds for a 2D array.
    arr: 2D array of lat or lon with shape (ny, nx)
    Returns: 2D array of bounds with shape (ny+1, nx+1)

    This function is used when constructing an xESMF grid, which requires
    cell-corner coordinates rather than cell centers. The bounds are 
    approximated by averaging adjacent grid cells and extrapolating at edges.
    """
    ny, nx = arr.shape

    # Compute interior midpoints (unused in final calculation, but preserves idea)
    b = 0.5 * (arr[:-1, :-1] + arr[1:, 1:])

    # Allocate full bounds array (one larger in each dimension)
    b_full = np.zeros((ny+1, nx+1))

    # Fill interior bounds using 4-cell average (standard finite-difference midpoint)
    b_full[1:-1, 1:-1] = 0.25 * (arr[:-1, :-1] + arr[1:, :-1] + arr[:-1, 1:] + arr[1:, 1:])

    # Top/bottom edges: extrapolate one half-grid outside domain
    b_full[0, 1:-1] = arr[0, :-1] - 0.5 * (arr[1, :-1] - arr[0, :-1])
    b_full[-1, 1:-1] = arr[-1, :-1] + 0.5 * (arr[-1, :-1] - arr[-2, :-1])

    # Left/right edges: similar 1D extrapolation in x
    b_full[1:-1, 0] = arr[:-1, 0] - 0.5 * (arr[:-1, 1] - arr[:-1, 0])
    b_full[1:-1, -1] = arr[:-1, -1] + 0.5 * (arr[:-1, -1] - arr[:-1, -2])

    # Fill corners by copying nearest interior bounds (simple but sufficient)
    b_full[0, 0] = b_full[1, 1]
    b_full[0, -1] = b_full[1, -2]
    b_full[-1, 0] = b_full[-2, 1]
    b_full[-1, -1] = b_full[-2, -2]

    return b_full

# Extract lat/lon from RACMO regridded Antarctic domain (2D arrays of cell centers)
lat = racmo["lat"].values
lon = racmo["lon"].values

# Compute corner bounds needed for conservative regridding in xESMF
lat_b = compute_2d_bounds(lat)
lon_b = compute_2d_bounds(lon)

# Build an xarray dataset describing the grid geometry for xESMF
grid_ds = xr.Dataset(
    {
        "lat": (["y", "x"], lat),
        "lon": (["y", "x"], lon),
        "lat_b": (["y_b", "x_b"], lat_b),
        "lon_b": (["y_b", "x_b"], lon_b),
    }
)

print(grid_ds)

# Compute grid-cell surface area from lat/lon using xESMF utilities (km²)
cellArea = xe.util.cell_area(grid_ds, earth_radius=6371.0)
threshold = 800  # smoothing threshold: cap extreme large cells (>800 km²)

# Copy cell-area array so modifications do not affect original
cellArea_fixed = cellArea.copy()

# Smooth cell-area anomalies row-by-row
for i in range(cellArea_fixed.shape[0]):
    # Compute row mean of all cells that are below threshold
    row_mean = cellArea_fixed[i, :].where(cellArea_fixed[i, :] <= threshold).mean()
    # Replace oversized cells with the representative row mean
    cellArea_fixed[i, :] = xr.where(cellArea_fixed[i, :] > threshold, row_mean, cellArea_fixed[i, :])

# RACMO SMB field (kg m⁻² per month) → squeeze height dimension
smb = racmo["smb"].squeeze("height")

# Aggregate SMB to annual totals (kg m⁻² yr⁻¹)
smb_ann = smb.groupby("time.year").sum(dim="time").sel(year=slice(1979, 2022))

# Reload lat/lon again (same arrays), ensuring consistent shape names
lat = racmo["lat"].values
lon = racmo["lon"].values

# Convert smoothed cell area (km²) into xarray DataArray (for broadcasting)
cell_area_da = xr.DataArray(cellArea_fixed, dims=("rlat", "rlon"))

# Convert SMB flux into total annual mass per cell (Gt contribution computed later)
smb_mass_ann = smb_ann * cell_area_da * 1_000_000  # km² → m² using 1e6

# Spatial integral of SMB over Antarctica (Gt yr⁻¹)
antarctic_smb_Gt_ann = smb_mass_ann.sum(dim=("rlat", "rlon")) / 1e12

# --- 1️⃣  Prepare core inputs ---

# Annual SMB field again (kg m⁻² yr⁻¹)
smb = racmo["smb"].squeeze("height")
smb_ann = smb.groupby("time.year").sum(dim="time").sel(year=slice(1979, 2022))

# Convert cell area from km² → m² for proper SMB mass conversion
cell_area_m2 = cellArea_fixed * 1e6
cell_area_da = xr.DataArray(cell_area_m2, dims=("rlat", "rlon"))

# Drainage basin mask, identifying grounded-ice hydrological basins (e.g., IMBIE)
basin_mask = mask["GroundedIce"]

# Replace NaNs with 0 so the “non-basin” category is explicit
basin_mask = basin_mask.where(~np.isnan(basin_mask), 0)

# --- 2️⃣  Compute SMB mass (kg) per grid cell, each year ---

smb_mass_ann = smb_ann * cell_area_da

# --- 3️⃣  Flatten for vectorized grouped summation over basins ---

nlat, nlon = basin_mask.shape
basin_flat = basin_mask.values.reshape(nlat * nlon)

# Valid grounded-ice cells only
valid_mask = (basin_flat > 0) & np.isfinite(basin_flat)
basin_ids = basin_flat[valid_mask].astype(int)

# Prepare output DataFrame: index=years, columns=basin IDs
years = smb_ann["year"].values
basin_ids_unique = np.unique(basin_ids)
smb_per_basin_Gt = pd.DataFrame(index=years, columns=basin_ids_unique)

# --- 4️⃣  Loop over years: compute basin-summed SMB in gigatonnes ---

for i, yr in enumerate(years):
    # Flatten annual SMB mass for this year
    smb_yr = smb_mass_ann.sel(year=yr).values.reshape(nlat * nlon)

    # Extract only grounded-ice cells
    smb_valid = smb_yr[valid_mask]

    # Organize into a DataFrame for easy groupby aggregation
    df = pd.DataFrame({"basin": basin_ids, "mass_kg": smb_valid})

    # Sum by basin and convert to Gt
    smb_sum = df.groupby("basin")["mass_kg"].sum() / 1e12
    smb_per_basin_Gt.loc[yr, smb_sum.index] = smb_sum.values

# --- 5️⃣  Finished computing SMB per basin (Gt yr⁻¹) ---
# print(smb_per_basin_Gt)


<xarray.Dataset> Size: 2MB
Dimensions:  (y: 240, x: 262, y_b: 241, x_b: 263)
Dimensions without coordinates: y, x, y_b, x_b
Data variables:
    lat      (y, x) float64 503kB -46.75 -46.92 -47.09 ... -47.42 -47.24 -47.07
    lon      (y, x) float64 503kB -126.9 -127.1 -127.3 ... 52.84 53.03 53.23
    lat_b    (y_b, x_b) float64 507kB -46.91 -46.67 -46.84 ... -47.17 -47.24
    lon_b    (y_b, x_b) float64 507kB -126.8 -127.0 -127.2 ... 52.69 52.89 53.28


In [20]:
# Importing ERA5 data, metadata about ice cores
era = xr.open_dataset(home_path + path + "/ae3baa6a74f0aa315dc3de6f83298f0e.nc")

# pr has units m -> convert to mm / kg m^2
pr = era["tp"] * 1000


lats_era = era["latitude"].values
lons_era = era["longitude"].values

# Compute number of days in each month (xarray provides days_in_month)
days_in_month = pr["valid_time"].dt.days_in_month
seconds_in_month = days_in_month 

# Convert monthly mean rate (mm/s) → monthly total (mm)
pr_monthly_total = pr * seconds_in_month

# Restrict to 1979–2000
pr_sel = pr_monthly_total.sel(valid_time=slice("1979-01-01", "2022-12-31"))

# Group by year and sum over months
pr_annual = pr_sel.groupby("valid_time.year").sum("valid_time")

# pr_annual is now ERA5 (year, lat, lon) with units mm/year
print(pr_annual.shape)

# Mapping the ice core locations to ERA5 indices
prCoords = pd.read_csv(home_path + path + "/AccumCoresCoords_NoBruce.csv", header=None)
coresLat = prCoords.iloc[:, 2].values
coresLonFirst = prCoords.iloc[:, 3].values
coresLon = (coresLonFirst + 360) % 360

# Select a numeric variable from your Dataset, e.g., 'tp' (replace with your variable name)
era_var = era["tp"]  # replace 'tp' with your actual variable name

lat_idx = []
lon_idx = []
grid_lats = []
grid_lons = []

for la, lo in zip(coresLat, coresLon):
    # Compute distances to all grid points
    lat_diff = np.abs(lats_era - la)[:, None]  # (n_lat, 1)
    lon_diff = np.abs(lons_era - lo)[None, :]  # (1, n_lon)
    dist = np.sqrt(lat_diff**2 + lon_diff**2)

    # Mask NaNs in the first valid_time slice
    mask = ~np.isnan(era_var.isel(valid_time=0).values)  # (n_lat, n_lon)
    dist_masked = np.where(mask, dist, np.inf)

    # Find indices of nearest valid grid cell
    i_lat, i_lon = np.unravel_index(np.argmin(dist_masked), dist_masked.shape)

    grid_lats.append(float(lats_era[i_lat]))
    grid_lons.append(float(lons_era[i_lon]))
    lat_idx.append(i_lat)
    lon_idx.append(i_lon)

# Store results back in DataFrame
prCoords["lat_idx"] = lat_idx
prCoords["lon_idx"] = lon_idx
prCoords["grid_lat"] = grid_lats
prCoords["grid_lon"] = grid_lons

print(prCoords.head())

accumCoordsInd = prCoords.iloc[:, 2:4].to_numpy()


(44, 301, 3600)
                       0                       1        2       3  lat_idx  \
0  vrs-13 (vostok stack)  Vostok composite VRS13 -78.4700  106.83      185   
1              B31 3.43W           B31Site DML07 -75.5800   -3.43      179   
2           B32 0.00667E           B32Site DML05 -75.0000   -0.01      179   
3            B33 6.4983E           B33Site DML17 -75.1700    6.50      152   
4              FB96DML01               FB96DML01 -74.8583   -2.55      179   

   lon_idx  grid_lat  grid_lon  
0     2868     -78.5     106.8  
1     3599     -77.9     179.9  
2     3599     -77.9     179.9  
3     1865     -75.2       6.5  
4     3599     -77.9     179.9  


In [23]:
# assuming precip_1 is a DataArray with dims ('time', 'lat', 'lon')
da_src = xr.DataArray(
    pr_annual,
    dims=["time", "lat", "lon"],
    coords={"lat": lats_era, "lon": lons_era}
)

# Build source grid dict (regular grid)
grid_src = {
    "lon": lons_era,
    "lat": lats_era
}

In [28]:
ds_out = xr.Dataset(
    {
        "lat": (["y", "x"], lat),
        "lon": (["y", "x"], lon),
    }
)

In [29]:
regridder = xe.Regridder(
    da_src,
    ds_out,
    method="bilinear",
    filename="regridder_file.nc",
    reuse_weights=False
)

In [30]:
result = regridder(da_src)

In [31]:
print(result.shape)

(44, 240, 262)
